In [30]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    # print()


# ================== 主逻辑 ==================
df = pd.read_csv("./history_results/results8/result_summary.csv")
# df = pd.read_csv("./results/result_summary.csv")

# df = df[df['Metric'].isin(['Recall', 'NDCG'])]
df = df[df['Campus'].isin(['campus_15'])]

loss_list = [f"loss{i}" for i in range(1, 6)]
loss_list = [f"loss{i}" for i in [3,5]]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 10"]

value = 'AbsDiff'
# value = "RelDiff(%)"

# ========== 生成表 (平均提升百分比) ==========
def make_tables(df, label):
    for topk in topk_list:
        df_topk = df[df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])[value]
            .mean()
            .reset_index()
        )
        table = pivot.pivot(index="Loss", columns="Model", values=value).round(3)
        print_colored_table(table, f"\n=== {label} | {topk} ===")


# print("========== 汇总表 ==========")

# 遍历 campus
for campus, df_c in df.groupby("Campus"):
    # print(f"\n########## Campus: {campus} ##########")
    make_tables(df_c, f"Results | Campus={campus}")



=== Results | Campus=campus_15 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss3         -0.001     0.001     0.002    -0.005
loss5         -0.001     0.000     0.001    -0.004


In [31]:
# ================== 主逻辑 ==================

# 确保数值列为数值类型
df[value] = pd.to_numeric(df[value], errors="coerce")

# 只考虑 loss3 和 loss5
loss_keep = ["loss3", "loss5"]
# loss_keep = ["loss3"]
topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 10"]

# print("========== 汇总表 ==========")

# 过滤数据
subset = df[df["Loss"].isin(loss_keep)]

# 遍历 (Campus, Loss, TopK)，分别打印表格
for campus, df_c in subset.groupby("Campus", sort=True):
    print(f"\n########## Campus: {campus} ##########")
    for loss in loss_keep:
        # print(f"###### Loss: {loss} ######")
        df_l = df_c[df_c["Loss"] == loss]
        for topk in topk_list:
            g = df_l[df_l["TopK"] == topk]
            if g.empty:
                continue
            pivot = (
                g.groupby(["Metric", "Model"])[value]
                 .mean()
                 .reset_index()
                 .pivot(index="Metric", columns="Model", values=value)
                 .sort_index()
                 .round(3)
            )
            pivot = pivot.reindex(sorted(pivot.columns), axis=1)  # 模型列排序
            print_colored_table(
                pivot,
                f"=== Results | Campus={campus} | Loss={loss} | TopK={topk} ==="
            )



########## Campus: campus_15 ##########
=== Results | Campus=campus_15 | Loss=loss3 | TopK=Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
Hit Ratio      0.000     0.002     0.001     0.016
NDCG          -0.003     0.001     0.002    -0.012
Precision      0.000     0.001     0.001     0.008
Recall        -0.001     0.002     0.005    -0.033
=== Results | Campus=campus_15 | Loss=loss5 | TopK=Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
Hit Ratio     -0.001     0.001    -0.003     0.017
NDCG          -0.001    -0.001     0.002    -0.012
Precision     -0.000     0.000    -0.002     0.008
Recall        -0.000     0.001     0.008    -0.031


In [32]:
import pandas as pd
from colorama import Fore, Style

for model, subdf in df.groupby("Model"):
    print(f"\n===== Model: {model} =====")

    # 先转成字符串表格
    table_str = subdf.to_string(index=False)

    # 按行拆分
    lines = table_str.split("\n")

    # 第一行是表头，原样打印
    print(lines[0])

    # 从第二行开始逐行处理
    for i, (_, row) in enumerate(subdf.iterrows(), start=1):
        line = lines[i]
        if row["RelDiff(%)"] > 0:
            print(Fore.RED + line + Style.RESET_ALL)
        else:
            print(line)



===== Model: NCL =====
Model    Campus  Loss   TopK    Metric  Baseline(loss0)   Value  AbsDiff  RelDiff(%)
  NCL campus_15 loss1  Top 3 Hit Ratio          0.14967 0.14229 -0.00738   -4.930848
  NCL campus_15 loss1  Top 3 Precision          0.24173 0.22981 -0.01192   -4.931121
  NCL campus_15 loss1  Top 3    Recall          0.24962 0.24251 -0.00711   -2.848329
  NCL campus_15 loss1  Top 3      NDCG          0.32576 0.30797 -0.01779   -5.461076
  NCL campus_15 loss1  Top 5 Hit Ratio          0.21565 0.20778 -0.00787   -3.649432
  NCL campus_15 loss1  Top 5 Precision          0.20898 0.20135 -0.00763   -3.651067
  NCL campus_15 loss1  Top 5    Recall          0.31966 0.31338 -0.00628   -1.964587
  NCL campus_15 loss1  Top 5      NDCG          0.34395 0.32868 -0.01527   -4.439599
  NCL campus_15 loss1 Top 10 Hit Ratio          0.34617 0.33482 -0.01135   -3.278736
  NCL campus_15 loss1 Top 10 Precision          0.16773 0.16223 -0.00550   -3.279079
  NCL campus_15 loss1 Top 10    Recall   